In [2]:
import torch
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
mens = open('data/men.txt').read().splitlines()
womens = open('data/women.txt').read().splitlines()

df = pd.DataFrame(mens + womens, columns=['name'])

df['name'] = df['name'].str.lower()
df = df[~df['name'].str.endswith('.')]
df = df[df['name'].str.len() >= 3]
df = df.drop_duplicates().reset_index(drop=True)
df = df.sample(frac=1.0).reset_index(drop=True) # shuffle

In [4]:
df.describe()

,name
count,23184
unique,23184
top,søren
freq,1


In [5]:
names = df['name'].to_list()
names[:10]

['søren',
 'kitti',
 'anttooni',
 'anni-marja',
 'jooni',
 'hetastiina',
 'epp',
 'esmee',
 'benedikta',
 'anneliina']

In [6]:
vocabulary = sorted(list(set(''.join(names))))
vocabulary.insert(0, '<S>')
vocabulary.insert(1, '<E>')
len(vocabulary)

54

In [7]:
atoi = {c: i for i, c in enumerate(vocabulary)}
itoa = {i: c for i, c in enumerate(vocabulary)}
atoi

{'<S>': 0,
 '<E>': 1,
 "'": 2,
 '-': 3,
 'a': 4,
 'b': 5,
 'c': 6,
 'd': 7,
 'e': 8,
 'f': 9,
 'g': 10,
 'h': 11,
 'i': 12,
 'j': 13,
 'k': 14,
 'l': 15,
 'm': 16,
 'n': 17,
 'o': 18,
 'p': 19,
 'q': 20,
 'r': 21,
 's': 22,
 't': 23,
 'u': 24,
 'v': 25,
 'w': 26,
 'x': 27,
 'y': 28,
 'z': 29,
 'à': 30,
 'á': 31,
 'â': 32,
 'ã': 33,
 'ä': 34,
 'å': 35,
 'ç': 36,
 'è': 37,
 'é': 38,
 'ê': 39,
 'ë': 40,
 'í': 41,
 'î': 42,
 'ï': 43,
 'ò': 44,
 'ó': 45,
 'ô': 46,
 'õ': 47,
 'ö': 48,
 'ø': 49,
 'ù': 50,
 'ú': 51,
 'ü': 52,
 'þ': 53}

In [8]:
block_size = 8 # context length: how many characters do we take to predict the next one?

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * block_size
    for ch in w:
      ix = atoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix] # crop and append
    context = context[1:] + [1] # add <E> at the end of the word

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  print(X.shape, Y.shape)
  return X, Y

n1 = int(0.8*len(names))
n2 = int(0.9*len(names))
Xtr,  Ytr  = build_dataset(names[:n1])     # 80%
Xdev, Ydev = build_dataset(names[n1:n2])   # 10%
Xte,  Yte  = build_dataset(names[n2:])     # 10%

torch.Size([128070, 8]) torch.Size([128070])
torch.Size([16098, 8]) torch.Size([16098])
torch.Size([15905, 8]) torch.Size([15905])


In [9]:
for x,y in zip(Xtr[:20], Ytr[:20]):
  print(''.join(itoa[int(ix.item())] for ix in x), '-->', itoa[int(y.item())])

<S><S><S><S><S><S><S><S> --> s
<S><S><S><S><S><S><S>s --> ø
<S><S><S><S><S><S>sø --> r
<S><S><S><S><S>sør --> e
<S><S><S><S>søre --> n
<S><S><S><S><S><S><S><S> --> k
<S><S><S><S><S><S><S>k --> i
<S><S><S><S><S><S>ki --> t
<S><S><S><S><S>kit --> t
<S><S><S><S>kitt --> i
<S><S><S><S><S><S><S><S> --> a
<S><S><S><S><S><S><S>a --> n
<S><S><S><S><S><S>an --> t
<S><S><S><S><S>ant --> t
<S><S><S><S>antt --> o
<S><S><S>antto --> o
<S><S>anttoo --> n
<S>anttoon --> i
<S><S><S><S><S><S><S><S> --> a
<S><S><S><S><S><S><S>a --> n


In [28]:
vocab_size = len(vocabulary)
n_embd = 24
n_hidden = 128

In [214]:
class WaveNet(torch.nn.Module):
  def __init__(self, vocab_size, n_embd, n_hidden):
    super().__init__()
    self.n = 2
    self.embedding = torch.nn.Embedding(vocab_size, n_embd)
    self.linear1 = torch.nn.Linear(n_embd * 2, n_hidden, bias=False)
    self.bn1 = torch.nn.BatchNorm1d(n_hidden*4)
    self.linear2 = torch.nn.Linear(n_hidden*2, n_hidden, bias=False)
    self.bn2 = torch.nn.BatchNorm1d(n_hidden*2)
    self.linear3 = torch.nn.Linear(n_hidden*2, n_hidden, bias=False)
    self.bn3 = torch.nn.BatchNorm1d(n_hidden*4)
    self.linear4 = torch.nn.Linear(n_hidden, vocab_size)

  def forward(self, x):
    x = self.embedding(x)

    x = self._flatten_consecutive(x)
    x = self.linear1(x)
    x = self._batchnorm(x, self.bn1)
    x = torch.tanh(x)

    x = self._flatten_consecutive(x)
    x = self.linear2(x)
    x = self._batchnorm(x, self.bn2)
    x = torch.tanh(x)

    x = self._flatten_consecutive(x)
    x = self.linear3(x)
    x = self._batchnorm(x, self.bn3)
    x = torch.tanh(x)

    logits = self.linear4(x)

    return logits
  
  def _batchnorm(self, x, bn):
    start_shape = x.shape

    if len(x.shape) == 3:
      _, T, C = x.shape
      x = bn(x.view(-1, T*C))

    x = x.view(start_shape)
    assert x.shape == start_shape, f"{x.shape} vs {start_shape}"

    return x
  
  def _flatten_consecutive(self, x):
    B, T, C = x.shape
    x = x.view(B, T//self.n, C*self.n)
    if x.shape[1] == 1:
      x = x.squeeze(1)
    
    return x

In [215]:
model = WaveNet(vocab_size, n_embd, n_hidden)

In [216]:
model

WaveNet(
  (embedding): Embedding(54, 24)
  (linear1): Linear(in_features=48, out_features=128, bias=False)
  (bn1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear2): Linear(in_features=256, out_features=128, bias=False)
  (bn2): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear3): Linear(in_features=256, out_features=128, bias=False)
  (bn3): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (linear4): Linear(in_features=128, out_features=54, bias=True)
)

In [217]:
model(Xtr[:5])

tensor([[-0.1479,  0.0140, -0.1784,  0.0524, -0.1130,  0.2124, -0.0169,  0.0142,
         -0.0413,  0.1895, -0.0901,  0.2178, -0.0309, -0.0048, -0.0446, -0.0508,
          0.0166, -0.2075,  0.1241,  0.2472,  0.1197,  0.1823,  0.0186,  0.0343,
          0.0799,  0.0742,  0.1272,  0.1492,  0.0122,  0.0683,  0.2059,  0.2638,
          0.0422, -0.1232,  0.0807, -0.2988, -0.2648,  0.0501, -0.0681,  0.2132,
          0.2737,  0.2790, -0.1691, -0.0431, -0.1325, -0.0666,  0.0661, -0.1078,
         -0.2338,  0.0969, -0.0525, -0.1051, -0.0198,  0.2014],
        [-0.1761, -0.1240,  0.1584,  0.1579,  0.0813,  0.0289,  0.1769,  0.0653,
         -0.1872,  0.0691,  0.0987, -0.0392,  0.2456, -0.1325, -0.1037, -0.0183,
          0.1210, -0.2702, -0.0313,  0.1701, -0.0829, -0.0499, -0.3963, -0.1512,
          0.0923,  0.1920,  0.2137,  0.1058, -0.1966,  0.1368, -0.0205, -0.2139,
         -0.0828, -0.0553,  0.1979,  0.0324,  0.0086, -0.1640,  0.0623,  0.0335,
          0.0175,  0.2601, -0.2072,  0.0631, 

In [220]:
from torch.nn import functional as F

probs = F.softmax(model(Xtr[:5]), dim=1)
pred = probs.max(dim=1).indices
pred

tensor([41, 52, 53, 42,  2])

In [221]:
for p in pred:
  print(itoa[int(p.item())])

í
ü
þ
î
'
